# Classifying Typeface Styles with Elliptic Fourier Descriptors

**Pipeline orchestration notebook**

This notebook walks through the five-stage pipeline end-to-end. Run cells in order. All real work is done by the modules in `../src/`; this notebook is the narration plus the orchestration.

## What this pipeline does

1. **Corpus** — load `fonts.csv` and confirm every font file exists and is readable.
2. **Outlines** — for each (font, lowercase letter a-z) pair, extract the outer contour as 200 (x, y) points equally spaced along arc length.
3. **EFD features** — compress each contour into a fixed-length vector using Elliptic Fourier Descriptors, at six harmonic orders (5, 10, 15, 20, 30, 40).
4. **Classification** — three classifiers (nearest centroid, random forest, PCA+logistic regression) under two cross-validation protocols (per-glyph stratified, per-font GroupKFold), with bootstrap 95% confidence intervals on macro-F1.
5. **Visualization** — four publication-ready figures plus a confusion matrix.

## What the calibration run found (20 fonts, 5 per class)

- Random forest macro-F1 = **0.40** at N=20 under per-font GroupKFold (vs 0.25 chance).
- Per-glyph stratified CV inflates this to 0.61 because 26 glyphs from the same font are not independent samples — confirming why per-font CV is the more honest evaluation.
- Accuracy DECLINES with more harmonics on the small corpus (50% at N=5 to 31% at N=40). At full 30-fonts-per-class scale this curve should rise then plateau.
- Per-class recall: serif 61%, calligraphic 48%, sans-serif 48%, display 15% (often confused for calligraphic — Lobster and Pacifico drive this).

All numbers are reproducible with `random_state=42`.

## Setup

In Google Colab, run the cell below to install dependencies and clone the Google Fonts repo. Locally, you can skip these if you've already installed via `requirements.txt`.

In [ ]:
# --- COLAB SETUP (skip if running locally) ---
# !pip install -q fonttools pyefd scikit-learn matplotlib seaborn pandas numpy scipy
# !git clone --depth 1 https://github.com/google/fonts.git ../data/google-fonts

# --- LOCAL SETUP ---
import sys, os
from pathlib import Path
# Add project root to path so we can import src/
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # so relative paths in fonts.csv resolve correctly
print('Working directory:', os.getcwd())

## Stage 1: Corpus validation

Reads `fonts.csv`, resolves filepaths against the project root, and confirms every font file is on disk and readable by fontTools. The summary should show 20 fonts (5 per class), all exist, all readable.

**If you see MISSING FILES:** the path in `fonts.csv` doesn't match what's in `data/google-fonts/`. Re-clone the Google Fonts repo or correct the path.

In [ ]:
from src import corpus

df = corpus.run('fonts.csv', fonts_root='.')
df.head()

## Stage 2: Glyph outline extraction

For each (font, glyph) pair, we use a custom fontTools pen to walk the glyph's Bezier description and produce a list of (x, y) points. Then we:

1. Select the **outer contour** by picking the sub-path with the largest absolute signed area (so 'a', 'b', 'd', 'e', 'g', 'o', 'p', 'q' use their outer ring, not the inner hole).
2. **Resample to 200 points equally spaced along arc length** — important because Bezier `t`-uniform sampling produces non-uniform arc-length spacing, which would distort the low harmonics in Stage 3.

Expected output: 520 glyph contours (20 fonts × 26 lowercase letters), zero failures.

In [ ]:
from src import outlines

outlines_dict, failures = outlines.run(df, output_path='outputs/outlines.pkl')

# Sanity check: show one extracted contour
import matplotlib.pyplot as plt
contour = outlines_dict['Lora']['a']
fig, ax = plt.subplots(figsize=(4, 4))
ax.plot(contour[:, 0], contour[:, 1], 'o-', ms=2)
ax.set_aspect('equal'); ax.set_title("Lora 'a' — outer contour (200 points)")
plt.show()

## Stage 3: EFD feature extraction

Each (n_points, 2) contour is passed to `pyefd.elliptic_fourier_descriptors` at six harmonic orders. After normalization, the first three coefficients become constants (1.0, 0.0, 0.0) and are dropped, giving feature dimensions of:

| Harmonic order N | Feature dimension |
|---|---|
| 5  | 17  |
| 10 | 37  |
| 15 | 57  |
| 20 | 77  (PRD primary) |
| 30 | 117 |
| 40 | 157 |

All six matrices are saved in a single `.npz` so Stage 4 (classification) and Stage 5 (sensitivity curve) can re-use them without recomputation.

In [ ]:
from src import efd

label_map = dict(zip(df.font_name, df.style_class))
feats = efd.run(outlines_dict, label_map, output_path='outputs/efd_features.npz')

## Stage 4: Classification

Three classifiers (nearest centroid, random forest, PCA+logistic regression) × two cross-validation protocols:

- **Per-glyph stratified k-fold** — the PRD's stated protocol. Treats each (font, glyph) pair as independent.
- **Per-font GroupKFold** — more honest. All 26 glyphs from one font stay together in train OR test, never both. This number is what we report as the headline because the per-glyph version inflates accuracy via within-font correlation.

Bootstrap 95% confidence interval on macro-F1 by resampling held-out fold predictions 1000 times.

We also run a sensitivity sweep: random forest under group CV at every harmonic order, used to draw Figure 3.

In [ ]:
from src import classify

results = classify.run(feats, output_path='outputs/classification_results.json')

# Headline table
import pandas as pd
rows = []
for clf_name, by_cv in results['main'].items():
    for cv_type, r in by_cv.items():
        rows.append({'classifier': clf_name, 'cv': cv_type,
                     'macro-F1': f"{r['macro_f1']:.3f}",
                     '95% CI': f"[{r['ci_low_95']:.3f}, {r['ci_high_95']:.3f}]"})
pd.DataFrame(rows)

## Stage 5: Visualization

Four PRD figures plus a confusion matrix diagnostic, all saved as 300 DPI PNG to `outputs/figures/`.

- **Fig 1 — PCA scatter.** EFD vectors for glyphs 'a' and 'g' projected to PC1 × PC2, colored by class.
- **Fig 2 — Harmonic reconstruction grid.** 4 classes × 5 harmonic orders showing how 'a' is rebuilt from progressively more ellipses.
- **Fig 3 — Accuracy vs harmonics.** Macro-F1 with bootstrap CI error bars for N = 5, 10, 15, 20, 30, 40.
- **Fig 4 — Feature importance.** RF importance aggregated by harmonic number — which harmonics carry the discriminative signal.
- **Diag — Confusion matrix.** Which classes get confused with which.

In [ ]:
from src import visualize

paths = visualize.run(feats, outlines_dict, label_map, results,
                       output_dir='outputs/figures')

# Display them inline
from IPython.display import Image, display
for key, path in paths.items():
    print(f'{key}: {path}')
    display(Image(path))

## What to do next

1. **Audit the figures.** Does the PCA scatter cluster classes? Does the accuracy-vs-harmonics curve match the PRD's prediction? Does the confusion matrix tell you which classes are easy/hard?
2. **Scale to 120 fonts.** Add 25 more rows per class to `fonts.csv` (see `labeling_protocol.md` for how to label). Re-run this notebook. No code changes needed.
3. **Write the paper.** Use the artifacts in `outputs/` to fill in Methods (cite `outlines.py: select_outer_contour` and the arc-length step), Results (Figs 1-4 plus the confusion matrix), and Discussion (the sensitivity curve's shape).

## Where to read about design choices

- Outer contour selection by signed area → `src/outlines.py: select_outer_contour`
- Arc-length resampling rationale → `src/outlines.py: resample_arc_length`
- Why feature dim is 4N-3 not 4N → `src/efd.py: compute_efd_features`
- Why we report both CV protocols → `src/classify.py: run`
- Bootstrap CI implementation → `src/classify.py: bootstrap_macro_f1`